# 01 — EDA: Financial PhraseBank

**Goal:** understand the labelled training corpus before any modelling.

We look at three things that drive every modelling choice downstream:

1. **Class balance** — sentiment data is almost always neutral-heavy.
2. **Text length distribution** — sets a sensible `max_length` for FinBERT and tells us whether TF-IDF can even see the same context.
3. **Label examples** — sanity-checks the dataset's labelling philosophy.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from config import LABELS, SPLITS_DIR
from data.load import load_phrasebank_splits

splits = load_phrasebank_splits()
for name, df in splits.items():
    print(name, df.shape)
splits['train'].head()

## Class balance per split

We stratified, so the proportions should match across train/val/test. Big imbalance toward neutral is exactly why the TF-IDF baseline uses `class_weight='balanced'`.

In [ ]:
rows = []
for split, df in splits.items():
    counts = df['label_str'].value_counts(normalize=True)
    for label in LABELS:
        rows.append({'split': split, 'label': label, 'share': float(counts.get(label, 0.0))})
share_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(data=share_df, x='label', y='share', hue='split', ax=ax)
ax.set_title('Class share by split (stratified 70/10/20)')
ax.set_ylabel('share of split')
plt.tight_layout(); plt.show()

### Why `class_weight='balanced'` matters here

The Financial PhraseBank `sentences_allagree` split is dominated by **neutral** sentences (often >55%). A logistic regression trained with the default uniform weighting will minimise its loss by simply over-predicting neutral, which inflates accuracy but tanks per-class F1 for negative — exactly the class an investor cares about most.

`class_weight='balanced'` rescales the loss inversely to class frequency, so each class contributes equally to the gradient. In practice this usually trades a small amount of overall accuracy for a much higher **macro-F1**, which is the metric we report in the leaderboard.

## Text length distribution

FinBERT's tokenizer caps inputs at 512 sub-word tokens. We can see how aggressive any truncation will be by looking at word counts (sub-word counts are roughly 1.3× word counts for English finance text).

In [ ]:
lengths = splits['train']['text'].str.split().str.len()
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.hist(lengths, bins=40, color='steelblue', edgecolor='white')
ax.axvline(lengths.median(), color='red', ls='--', label=f'median={int(lengths.median())} words')
ax.set_title('Word count per training sentence')
ax.set_xlabel('word count'); ax.set_ylabel('count'); ax.legend()
plt.tight_layout(); plt.show()
lengths.describe().round(1)

## A few examples per class

These are the *all-agree* labels: every annotator agreed on the same class. Useful for spotting the kinds of phrasing each class hinges on.

In [ ]:
for label in LABELS:
    print(f'\n--- {label.upper()} ---')
    sample = splits['train'][splits['train']['label_str'] == label].sample(3, random_state=42)
    for _, row in sample.iterrows():
        print('-', row['text'])